# Self-Driving Project: Exploratory Data Analysis (EDA)

This notebook contains the complete code for performing EDA on video data for lane following and object detection tasks. It covers video profiling, image quality analysis, lane detection, and motion-based object analysis.

In [ ]:
import cv2
import numpy as np
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import subprocess

# Setup paths
VIDEO_PATH = 'test_video.mp4'
FRAME_DIR = 'frames/'
os.makedirs(FRAME_DIR, exist_ok=True)

print("Libraries imported and environment setup.")

## 1. Video Profiling and Frame Extraction

We first extract metadata and frames from the video at a rate of 1 frame per second for analysis.

In [ ]:
# Extract video metadata using ffprobe
def get_video_metadata(path):
    cmd = f"ffprobe -v error -select_streams v:0 -show_entries stream=width,height,avg_frame_rate,duration,nb_frames -of default=noprint_wrappers=1 {path}"
    result = subprocess.check_output(cmd, shell=True).decode('utf-8')
    print("Video Metadata:\n", result)

# Extract frames at 1 fps
def extract_frames(path, output_dir):
    cmd = f"ffmpeg -i {path} -vf 'fps=1' {output_dir}/frame_%03d.jpg"
    subprocess.call(cmd, shell=True)
    print(f"Frames extracted to {output_dir}")

get_video_metadata(VIDEO_PATH)
extract_frames(VIDEO_PATH, FRAME_DIR)

## 2. Image Quality Analysis

Analyzing brightness, contrast, sharpness, and color distribution.

In [ ]:
def analyze_image_quality(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    brightness = np.mean(gray)
    contrast = np.std(gray)
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
    
    return {
        'filename': os.path.basename(image_path),
        'brightness': brightness,
        'contrast': contrast,
        'sharpness': sharpness,
        'mean_b': np.mean(img[:, :, 0]),
        'mean_g': np.mean(img[:, :, 1]),
        'mean_r': np.mean(img[:, :, 2])
    }

image_paths = sorted(glob.glob(os.path.join(FRAME_DIR, '*.jpg')))
quality_results = [analyze_image_quality(p) for p in image_paths]
df_quality = pd.DataFrame([r for r in quality_results if r])

# Plotting
plt.figure(figsize=(15, 10))
plt.subplot(2, 2, 1); sns.lineplot(data=df_quality, x=df_quality.index, y='brightness'); plt.title('Brightness')
plt.subplot(2, 2, 2); sns.lineplot(data=df_quality, x=df_quality.index, y='contrast'); plt.title('Contrast')
plt.subplot(2, 2, 3); sns.lineplot(data=df_quality, x=df_quality.index, y='sharpness'); plt.title('Sharpness')
plt.subplot(2, 2, 4);
sns.lineplot(data=df_quality, x=df_quality.index, y='mean_r', color='red', label='Red')
sns.lineplot(data=df_quality, x=df_quality.index, y='mean_g', color='green', label='Green')
sns.lineplot(data=df_quality, x=df_quality.index, y='mean_b', color='blue', label='Blue')
plt.title('Color Channels'); plt.legend(); plt.tight_layout(); plt.show()

## 3. Lane Detection Analysis

Using edge detection and Hough lines within a Region of Interest (ROI).

In [ ]:
def analyze_lanes(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    
    height, width = img.shape[:2]
    roi_mask = np.zeros_like(img[:, :, 0])
    roi_vertices = np.array([[(0, height), (width, height), (width, height // 2), (0, height // 2)]], dtype=np.int32)
    cv2.fillPoly(roi_mask, roi_vertices, 255)
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 50, 150)
    masked_edges = cv2.bitwise_and(edges, roi_mask)
    
    lines = cv2.HoughLinesP(masked_edges, 1, np.pi/180, threshold=50, minLineLength=50, maxLineGap=100)
    
    return {
        'filename': os.path.basename(image_path),
        'lane_line_count': len(lines) if lines is not None else 0,
        'edge_density_roi': np.sum(masked_edges > 0) / (height * width / 2)
    }

lane_results = [analyze_lanes(p) for p in image_paths]
df_lane = pd.DataFrame([r for r in lane_results if r])

plt.figure(figsize=(10, 5))
plt.plot(df_lane.index, df_lane['lane_line_count'])
plt.title('Detected Lane Line Count over Time'); plt.xlabel('Frame'); plt.ylabel('Count'); plt.show()

## 4. Object Detection (Motion-based)

Using MOG2 background subtraction to detect moving objects.

In [ ]:
backSub = cv2.createBackgroundSubtractorMOG2()
motion_stats = []

for i, path in enumerate(image_paths):
    frame = cv2.imread(path)
    if frame is None: continue
    
    fgMask = backSub.apply(frame)
    moving_pixels = np.sum(fgMask > 0)
    motion_ratio = moving_pixels / (fgMask.shape[0] * fgMask.shape[1])
    
    contours, _ = cv2.findContours(fgMask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    large_contours = [cnt for cnt in contours if cv2.contourArea(cnt) > 500]
    
    motion_stats.append({
        'frame': i,
        'motion_ratio': motion_ratio,
        'potential_object_count': len(large_contours)
    })

df_motion = pd.DataFrame(motion_stats)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1); plt.plot(df_motion['frame'], df_motion['motion_ratio']); plt.title('Motion Ratio')
plt.subplot(1, 2, 2); plt.plot(df_motion['frame'], df_motion['potential_object_count']); plt.title('Object Count')
plt.tight_layout(); plt.show()